# Kata 17 — Procesamiento Masivo con Batches API

> Spec: [`specs/017-batch-processing/spec.md`](../../specs/017-batch-processing/spec.md)  |  Arquitectura: [`ARCHITECTURE.md`](../../ARCHITECTURE.md)

## Setup

Si `ANTHROPIC_API_KEY` no está en el entorno, se pedirá aquí (sin echo).

In [ ]:
from katas._shared.bootstrap import bootstrap

client, settings = bootstrap(budget_calls=4)
print("modelo:", settings.model)

## 1. Concepto en mis palabras

Para cargas no interactivas (auditorías, evaluaciones, backfills), la **Message Batches API** procesa lotes offline a ~50 % del costo. Cada request lleva un `custom_id` único que correlaciona request↔response y permite recuperación selectiva.

## 2. Por qué importa

Procesar 10 000 prompts en bucle síncrono paga tarifa real-time, pega rate limits, y no aísla fallos. Batch es el patrón correcto para cualquier carga que no bloquee al usuario.

## 3. Ejemplo correcto — submit + poll + correlate

> ⚠️ Esta celda **crea** un batch real (consume cuota). El batch se completa en minutos a horas. Ajusta `WAIT_SECONDS` o salta la espera.

In [ ]:
import time

DOCS = [
    {"id": "summ-1", "text": "El cliente reporta latencia >5s en checkout, ya pasó tres veces."},
    {"id": "summ-2", "text": "Pago rechazado en Atlas tier 2, error AUTH-503."},
    {"id": "summ-3", "text": "Solicitud de feature: integración con Slack para alertas."},
]

requests_payload = [
    {
        "custom_id": d["id"],
        "params": {
            "model": settings.model,
            "max_tokens": 200,
            "messages": [{"role": "user", "content": f"Resume en una línea: {d['text']}"}],
        },
    }
    for d in DOCS
]

batch = client.messages.batches.create(requests=requests_payload)
print("batch creado:", batch.id, "| status:", batch.processing_status)

# Polling sencillo (en producción usa webhooks o un job)
WAIT_SECONDS = 30
elapsed = 0
while batch.processing_status != "ended" and elapsed < 180:
    time.sleep(WAIT_SECONDS); elapsed += WAIT_SECONDS
    batch = client.messages.batches.retrieve(batch.id)
    print(f"  t+{elapsed}s status={batch.processing_status}")

print("\nrequest_counts:", batch.request_counts)

In [ ]:
if batch.processing_status == "ended":
    for r in client.messages.batches.results(batch.id):
        print(r.custom_id, "→", r.result.type)
        if r.result.type == "succeeded":
            msg = r.result.message
            print("   ", next((b.text for b in msg.content if b.type == "text"), "")[:120])
else:
    print("batch sigue en proceso; vuelve luego con client.messages.batches.retrieve(batch.id)")

### 3.1 Recuperación selectiva ante fallos parciales

In [ ]:
def reprocess_failed(batch_id):
    failed = []
    for r in client.messages.batches.results(batch_id):
        if r.result.type != "succeeded":
            failed.append(r.custom_id)
    print("fallidos:", failed)
    if not failed:
        print("nada que reprocesar")
        return None
    # Re-construir requests SOLO para los fallidos (re-usando el mismo custom_id)
    sub = [r for r in requests_payload if r["custom_id"] in failed]
    if sub:
        return client.messages.batches.create(requests=sub)
    return None

if batch.processing_status == "ended":
    reprocess_failed(batch.id)

## 4. Anti-patrón — bucle síncrono

In [ ]:
import time
def synchronous_loop(client, docs):
    out = []
    t0 = time.time()
    for d in docs:
        resp = client.messages.create(
            messages=[{"role":"user","content": f"Resume: {d['text']}"}],
        )
        out.append((d["id"], next((b.text for b in resp.content if b.type == "text"), "")[:80]))
    return out, time.time() - t0

# Atención: consume budget normal y paga tarifa real-time
sync_out, dt = synchronous_loop(client, DOCS[:2])  # sólo 2 para no quemar
print(f"síncrono: {dt:.1f}s, items: {len(sync_out)}")

En producción con 10 000 docs, el bucle síncrono:

- Tarda horas (vs minutos en batch).
- Paga tarifa full vs ~50 % en batch.
- Cualquier fallo requiere lógica de retry manual, sin recuperación selectiva.
- Los rate limits cortan a la mitad del trabajo.

## 5. Argumento de certificación

- **`custom_id` único** por request: correlación 1:1 entre input y output.
- **Polling vs webhook**: el ejemplo polea, en producción se usa webhook (`webhook_url=`).
- **Recuperación selectiva**: reprocesar sólo los fallidos, no el batch entero.
- **Conexión con otros katas**: cada request del batch puede ser una extracción defensiva (Kata 5) o un review (Kata 13); los errores aparecen como Kata 6.

## 6. Auto-evaluación

**1. ¿Cuál es el ahorro económico esperado vs API real-time?**

~50 % en input/output tokens (consultar pricing actual). Más el ahorro indirecto de no necesitar lógica de retry/backoff propia.

**2. ¿Cómo recupero un batch interrumpido sin re-pagar el 99 % exitoso?**

Filtrando los `result.type != "succeeded"` y armando un nuevo batch sólo con esos `custom_id`. El éxito previo no se re-procesa.

**3. ¿Qué hago si dos requests del batch tienen el mismo `custom_id` por error?**

La API lo rechaza al `create`. La defensa: validar localmente con `assert len({r["custom_id"] for r in reqs}) == len(reqs)` antes de enviar.